# NexaDoc-50k training (2×T4)

Trains the **new** pipeline on `kaggle.com/datasets/walidmostafa77/nexadoc-50k`.

- Stage 1: shadow removal, 256×256, MobileNetV2 IOANet.
- Stage 2: clean 256² → 768×1024 upsample, residual U-Net.
- Legacy `train.py` is **not** used.

Run the cells top to bottom. Re-running a training cell resumes from `last.pt`.


In [ ]:
# 1. Environment
import os, subprocess, sys, torch

print("torch", torch.__version__)
print("GPUs", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, "Select Kaggle accelerator: GPU T4 x2"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "tensorboard"], check=True)


In [ ]:
# 2. Code (branch nexa retrain, not main)
REPO = "https://github.com/walidmoustafa2077/DocShadow-Lite.git"
BRANCH = "nexadoc-retrain"

os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "DocShadow-Lite"], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO], check=True)
os.chdir("/kaggle/working/DocShadow-Lite")
print(subprocess.check_output(["git", "log", "-1", "--oneline"]).decode().strip())


In [ ]:
# 3. Dataset sanity
from pathlib import Path
DATA = Path("/kaggle/input/nexadoc-50k")
assert (DATA / "deshadow" / "train" / "input").is_dir(), DATA
for task in ("deshadow", "upsample"):
    counts = {s: len(list((DATA / task / s / "input").glob("*.jpg"))) for s in ("train", "val", "test")}
    print(task, counts)


## Stage 1 — deshadow 256²

40,000 train / 5,000 val. Batch 24 per GPU, 100 epochs, EMA, AMP.


In [ ]:
# 4. Stage 1 train (auto-resume)
OUT1 = "/kaggle/working/outputs/stage1"
os.makedirs(OUT1, exist_ok=True)
cmd = ["torchrun", "--nproc_per_node=2", "train_nexadoc.py",
       "--stage", "1", "--data-root", "/kaggle/input/nexadoc-50k",
       "--output-dir", OUT1]
last = os.path.join(OUT1, "last.pt")
if os.path.exists(last):
    cmd += ["--resume", last]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## Stage 2 — clean 256² → 768×1024

Independent of Stage 1. Trains only on clean `upsample/input`, pure L1.
7,961 train / 1,059 val. Batch 4 per GPU, 40 epochs.


In [ ]:
# 5. Stage 2 train (auto-resume)
OUT2 = "/kaggle/working/outputs/stage2"
os.makedirs(OUT2, exist_ok=True)
cmd = ["torchrun", "--nproc_per_node=2", "train_nexadoc.py",
       "--stage", "2", "--data-root", "/kaggle/input/nexadoc-50k",
       "--output-dir", OUT2]
last = os.path.join(OUT2, "last.pt")
if os.path.exists(last):
    cmd += ["--resume", last]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## Evaluate on the official test split

In [ ]:
# 6. Evaluation (runs whichever checkpoints exist)
if os.path.exists(os.path.join(OUT1, "best.pt")):
    subprocess.run(["python", "evaluate_nexadoc.py", "--stage", "1",
                    "--data-root", "/kaggle/input/nexadoc-50k",
                    "--checkpoint", os.path.join(OUT1, "best.pt"),
                    "--output-dir", os.path.join(OUT1, "eval"), "--save-samples", "8"], check=True)
if os.path.exists(os.path.join(OUT2, "best.pt")):
    subprocess.run(["python", "evaluate_nexadoc.py", "--stage", "2",
                    "--data-root", "/kaggle/input/nexadoc-50k",
                    "--checkpoint", os.path.join(OUT2, "best.pt"),
                    "--output-dir", os.path.join(OUT2, "eval"), "--save-samples", "8"], check=True)


In [ ]:
# 7. Package outputs for download / next session
import shutil
shutil.make_archive("/kaggle/working/nexadoc_results", "zip", "/kaggle/working", "outputs")
print("wrote /kaggle/working/nexadoc_results.zip")


## Plan / checks

1. Stage 1 first. Watch val PSNR/SSIM in `metrics.jsonl`; stop if val PSNR stalls for 20 epochs.
2. Stage 2 second. Beat the bicubic baseline: MAE 0.0377, PSNR 20.01, SSIM 0.835.
3. Evaluate on the official 5,000/980 test split; use `best.pt` (EMA).
4. Stage 2 is trained on clean inputs; at deployment feed Stage 1's 256² output.
5. Leakage caveat: splits are index-based, not document-disjoint. Do not publish test numbers as document-disjoint.
6. Stage 2 covers A4 only. Other page types are out of distribution.
7. Kaggle sessions: re-run training cells; `last.pt` auto-resumes. Keep `outputs/` in a Kaggle Dataset between sessions.
